In [ ]:
!pip install -q transformers datasets accelerate sentencepiece evaluate

In [ ]:
from dataclasses import dataclass
import os
import re
import ast
import json
import glob
import random
import numpy as np
import torch
import urllib.request
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    MT5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

@dataclass
class Config:
    model_name: str = "google/mt5-small"
    max_source_length: int = 192
    max_target_length: int = 128
    batch_size: int = 4
    eval_batch_size: int = 4
    epochs: int = 5
    lr: float = 3e-4
    seed: int = 42
    output_root: str = "/kaggle/working/absa_models"
    synthetic_dir: str = "/kaggle/input/datasets/anderpena/synthetic-data"

cfg = Config()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(cfg.output_root, exist_ok=True)

print("Device:", device)
print("Output root:", cfg.output_root)

Device: cuda
Output root: /kaggle/working/absa_models


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/Swaggy66/M-ABSA/main/data"

def download_split(domain, lang, split):
    url = f"{BASE_URL}/{domain}/{lang}/{split}.txt"
    with urllib.request.urlopen(url) as f:
        lines = f.read().decode("utf-8").strip().splitlines()
    return Dataset.from_list([{"text": line} for line in lines if line.strip()])

en_train_raw = download_split("restaurant", "en", "train")
en_dev_raw   = download_split("restaurant", "en", "dev")
en_test_raw  = download_split("restaurant", "en", "test")

es_train_raw = download_split("restaurant", "es", "train")
es_dev_raw   = download_split("restaurant", "es", "dev")
es_test_raw  = download_split("restaurant", "es", "test")

print("English:", len(en_train_raw), len(en_dev_raw), len(en_test_raw))
print("Spanish translated:", len(es_train_raw), len(es_dev_raw), len(es_test_raw))
print(en_train_raw[0]["text"])

English: 1264 316 544
Spanish translated: 1264 316 544
We have gone for dinner only a few times but the same great quality and service is given .####[['service', 'service general', 'positive'], ['dinner', 'food quality', 'positive']]


In [ ]:
def safe_parse_labels(labels_text):
    try:
        return ast.literal_eval(labels_text)
    except Exception:
        fixed = re.sub(r"\bNULL\b", "'NULL'", labels_text)
        try:
            return ast.literal_eval(fixed)
        except Exception:
            return []

def parse_example(example):
    text = example["text"]
    if "####" not in text:
        return {"sentence": text.strip(), "triplets": []}

    sentence, labels_text = text.split("####", 1)
    triplets = safe_parse_labels(labels_text.strip())

    cleaned = []
    for t in triplets:
        if isinstance(t, (list, tuple)) and len(t) == 3:
            term, cat, pol = t
            cleaned.append((str(term).strip(), str(cat).strip(), str(pol).strip()))

    return {
        "sentence": sentence.strip(),
        "triplets": cleaned
    }

en_train_parsed = en_train_raw.map(parse_example)
en_dev_parsed   = en_dev_raw.map(parse_example)
en_test_parsed  = en_test_raw.map(parse_example)

es_train_parsed = es_train_raw.map(parse_example)
es_dev_parsed   = es_dev_raw.map(parse_example)
es_test_parsed  = es_test_raw.map(parse_example)

print(en_train_parsed[0])

Map:   0%|          | 0/1264 [00:00<?, ? examples/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Map:   0%|          | 0/544 [00:00<?, ? examples/s]

Map:   0%|          | 0/1264 [00:00<?, ? examples/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Map:   0%|          | 0/544 [00:00<?, ? examples/s]

{'text': "We have gone for dinner only a few times but the same great quality and service is given .####[['service', 'service general', 'positive'], ['dinner', 'food quality', 'positive']]", 'sentence': 'We have gone for dinner only a few times but the same great quality and service is given .', 'triplets': [['service', 'service general', 'positive'], ['dinner', 'food quality', 'positive']]}


In [ ]:
def load_synthetic_examples(synth_dir):
    txt_files = glob.glob(os.path.join(synth_dir, "**", "*.txt"), recursive=True)
    jsonl_files = glob.glob(os.path.join(synth_dir, "**", "*.jsonl"), recursive=True)
    json_files = glob.glob(os.path.join(synth_dir, "**", "*.json"), recursive=True)

    examples = []

    if txt_files:
        for fp in txt_files:
            with open(fp, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line and "####" in line:
                        examples.append({"text": line})

    elif jsonl_files:
        for fp in jsonl_files:
            with open(fp, "r", encoding="utf-8") as f:
                for line in f:
                    obj = json.loads(line)
                    if "text" in obj:
                        examples.append({"text": obj["text"]})
                    elif "sentence" in obj and "triplets" in obj:
                        triplets = obj["triplets"]
                        text = obj["sentence"] + "####" + str(triplets)
                        examples.append({"text": text})

    elif json_files:
        for fp in json_files:
            with open(fp, "r", encoding="utf-8") as f:
                data = json.load(f)

            if isinstance(data, list):
                for obj in data:
                    if "text" in obj:
                        examples.append({"text": obj["text"]})
                    elif "sentence" in obj and "triplets" in obj:
                        text = obj["sentence"] + "####" + str(obj["triplets"])
                        examples.append({"text": text})

    return Dataset.from_list(examples)

syn_raw = load_synthetic_examples(cfg.synthetic_dir)
print("Synthetic raw examples:", len(syn_raw))

syn_parsed = syn_raw.map(parse_example)

print(syn_parsed[0] if len(syn_parsed) > 0 else "No synthetic examples found")

Synthetic raw examples: 1200


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

{'text': "El hotel está en una ubicación privilegiada, muy cerca del centro histórico de la ciudad.####[['ubicación', 'location general', 'positive']]", 'sentence': 'El hotel está en una ubicación privilegiada, muy cerca del centro histórico de la ciudad.', 'triplets': [['ubicación', 'location general', 'positive']]}


In [ ]:
syn_parsed = syn_parsed.shuffle(seed=cfg.seed)

split_idx = int(0.9 * len(syn_parsed))
syn_train_parsed = syn_parsed.select(range(split_idx))
syn_dev_parsed   = syn_parsed.select(range(split_idx, len(syn_parsed)))

print("Synthetic train/dev:", len(syn_train_parsed), len(syn_dev_parsed))

Synthetic train/dev: 1080 120


In [ ]:
def normalize_label(text):
    return str(text).strip().replace(" ", "").lower()

def format_triplets(triplets):
    pieces = []
    for term, cat, pol in triplets:
        term = str(term).strip()
        cat = normalize_label(cat)
        pol = normalize_label(pol)
        pieces.append(f"[TERM] {term} [CAT] {cat} [POL] {pol}")
    return " [SEP] ".join(pieces)

def format_seq2seq(example):
    return {
        "input_text": f"extract triplets: {example['sentence']}",
        "target_text": format_triplets(example["triplets"]),
        "gold_triplets": [
            (str(term).strip(), normalize_label(cat), normalize_label(pol))
            for term, cat, pol in example["triplets"]
        ]
    }

en_train_fmt = en_train_parsed.map(format_seq2seq)
en_dev_fmt   = en_dev_parsed.map(format_seq2seq)
en_test_fmt  = en_test_parsed.map(format_seq2seq)

es_train_fmt = es_train_parsed.map(format_seq2seq)
es_dev_fmt   = es_dev_parsed.map(format_seq2seq)
es_test_fmt  = es_test_parsed.map(format_seq2seq)

syn_train_fmt = syn_train_parsed.map(format_seq2seq)
syn_dev_fmt   = syn_dev_parsed.map(format_seq2seq)

print(en_train_fmt[0]["input_text"])
print(en_train_fmt[0]["target_text"])

Map:   0%|          | 0/1264 [00:00<?, ? examples/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Map:   0%|          | 0/544 [00:00<?, ? examples/s]

Map:   0%|          | 0/1264 [00:00<?, ? examples/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Map:   0%|          | 0/544 [00:00<?, ? examples/s]

Map:   0%|          | 0/1080 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

extract triplets: We have gone for dinner only a few times but the same great quality and service is given .
[TERM] service [CAT] servicegeneral [POL] positive [SEP] [TERM] dinner [CAT] foodquality [POL] positive


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=cfg.max_source_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["target_text"],
        max_length=cfg.max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = [
        [tok if tok != tokenizer.pad_token_id else -100 for tok in seq]
        for seq in labels["input_ids"]
    ]
    return model_inputs

def tokenize_dataset(ds):
    tok = ds.map(tokenize_function, batched=True)
    keep_cols = ["input_ids", "attention_mask", "labels"]
    tok = tok.remove_columns([c for c in tok.column_names if c not in keep_cols])
    return tok

en_train_tok = tokenize_dataset(en_train_fmt)
en_dev_tok   = tokenize_dataset(en_dev_fmt)

es_train_tok = tokenize_dataset(es_train_fmt)
es_dev_tok   = tokenize_dataset(es_dev_fmt)

syn_train_tok = tokenize_dataset(syn_train_fmt)
syn_dev_tok   = tokenize_dataset(syn_dev_fmt)

Map:   0%|          | 0/1264 [00:00<?, ? examples/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Map:   0%|          | 0/1264 [00:00<?, ? examples/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Map:   0%|          | 0/1080 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

In [ ]:
combined_train_fmt = Dataset.from_list(list(es_train_fmt) + list(syn_train_fmt))
combined_dev_fmt   = es_dev_fmt

combined_train_tok = tokenize_dataset(combined_train_fmt)
combined_dev_tok   = tokenize_dataset(combined_dev_fmt)

print("Combined train size:", len(combined_train_fmt))

Map:   0%|          | 0/2344 [00:00<?, ? examples/s]

Map:   0%|          | 0/316 [00:00<?, ? examples/s]

Combined train size: 2344


In [ ]:
def train_model(train_tok, dev_tok, output_dir, epochs=None):
    if epochs is None:
        epochs = cfg.epochs

    config = AutoConfig.from_pretrained(cfg.model_name)
    config.tie_word_embeddings = False

    model = MT5ForConditionalGeneration.from_pretrained(cfg.model_name, config=config)
    model = model.to(device)

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        return_tensors="pt"
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        learning_rate=cfg.lr,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.eval_batch_size,
        weight_decay=0.01,
        logging_steps=50,
        save_strategy="no",
        eval_strategy="epoch",
        predict_with_generate=True,
        generation_max_length=cfg.max_target_length,
        fp16=False,
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        data_collator=data_collator,
        processing_class=tokenizer
    )

    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    return model

In [ ]:
def parse_generated_triplets(text):
    if not text or not text.strip():
        return []

    text = text.replace("<pad>", "").replace("</s>", "").strip()
    chunks = [c.strip() for c in text.split("[SEP]") if c.strip()]
    triplets = []

    for chunk in chunks:
        m = re.search(
            r"\[TERM\]\s*(.*?)\s*\[CAT\]\s*(.*?)\s*\[POL\]\s*(.*)",
            chunk
        )
        if m:
            term = m.group(1).strip()
            cat = normalize_label(m.group(2))
            pol = normalize_label(m.group(3))
            triplets.append((term, cat, pol))

    return triplets

In [ ]:
def evaluate_model(model, formatted_ds, label="", max_examples_debug=5):
    model.eval()
    tp = fp = fn = 0
    debug_examples = []

    for i in range(len(formatted_ds)):
        ex = formatted_ds[i]

        inputs = tokenizer(
            ex["input_text"],
            return_tensors="pt",
            max_length=cfg.max_source_length,
            truncation=True
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=cfg.max_target_length,
                num_beams=4,
                early_stopping=True
            )

        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        pred_triplets = set(parse_generated_triplets(pred_text))
        gold_triplets = set(tuple(t) for t in ex["gold_triplets"])

        tp += len(pred_triplets & gold_triplets)
        fp += len(pred_triplets - gold_triplets)
        fn += len(gold_triplets - pred_triplets)

        if len(debug_examples) < max_examples_debug:
            debug_examples.append({
                "input": ex["input_text"],
                "gold": list(gold_triplets),
                "pred": list(pred_triplets),
                "raw_pred": pred_text
            })

        if i % 100 == 0:
            print(f"{label} -> evaluados {i}/{len(formatted_ds)}")

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    return {
        "label": label,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "debug_examples": debug_examples
    }

In [ ]:
model_a_dir = os.path.join(cfg.output_root, "model_a_en_real")
print("Training Model A - English real")
model_a = train_model(en_train_tok, en_dev_tok, model_a_dir)

Training Model A - English real


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,1.345918,0.645475
2,0.721996,0.483337
3,0.584403,0.416588
4,1.060446,0.381571
5,0.454915,0.380233


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
model_b_dir = os.path.join(cfg.output_root, "model_b_es_translated")
print("Training Model B - Spanish translated")
model_b = train_model(es_train_tok, es_dev_tok, model_b_dir)

Training Model B - Spanish translated


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,1.481503,0.687046
2,0.835191,0.575491
3,0.671283,0.485308
4,0.824243,0.471614
5,0.521381,0.462716


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
model_c_dir = os.path.join(cfg.output_root, "model_c_es_synthetic")
print("Training Model C - Spanish synthetic")
model_c = train_model(syn_train_tok, syn_dev_tok, model_c_dir)

Training Model C - Spanish synthetic


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,3.101540,0.501294
2,0.646984,0.369118
3,0.489891,0.323270
4,0.417297,0.302473
5,0.368632,0.286425


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
model_d_dir = os.path.join(cfg.output_root, "model_d_es_combined")
print("Training Model D - Spanish combined")
model_d = train_model(combined_train_tok, combined_dev_tok, model_d_dir)

Training Model D - Spanish combined


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,1.086113,0.651803
2,0.656695,0.567642
3,0.527507,0.493550
4,0.463846,0.493902
5,0.397985,0.469765


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from transformers import MT5ForConditionalGeneration, AutoTokenizer, AutoConfig

def load_model(model_dir):
    config = AutoConfig.from_pretrained(model_dir)
    model = MT5ForConditionalGeneration.from_pretrained(model_dir, config=config)
    model = model.to(device)
    return model

# Vuelve a crear tokenizer base si todavía no existe en esta sesión
if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

model_a_dir = os.path.join(cfg.output_root, "model_a_en_real")
model_b_dir = os.path.join(cfg.output_root, "model_b_es_translated")
model_c_dir = os.path.join(cfg.output_root, "model_c_es_synthetic")
model_d_dir = os.path.join(cfg.output_root, "model_d_es_combined")

# Carga solo los que tengas entrenados
model_a = load_model(model_a_dir)
model_b = load_model(model_b_dir)
# model_c = load_model(model_c_dir)  # descomenta si lo entrenaste
model_d = load_model(model_d_dir)

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
results = []

print("\nEvaluating Model A on English test")
res_a_en = evaluate_model(model_a, en_test_fmt, label="A_EN_on_EN")
results.append(res_a_en)

print("\nEvaluating Model A on Spanish test (cross-lingual)")
res_a_es = evaluate_model(model_a, es_test_fmt, label="A_EN_on_ES")
results.append(res_a_es)

print("\nEvaluating Model B on Spanish test")
res_b_es = evaluate_model(model_b, es_test_fmt, label="B_ES_translated_on_ES")
results.append(res_b_es)

print("\nEvaluating Model C on Spanish test")
res_c_es = evaluate_model(model_c, es_test_fmt, label="C_ES_synthetic_on_ES")
results.append(res_c_es)

print("\nEvaluating Model D on Spanish test")
res_d_es = evaluate_model(model_d, es_test_fmt, label="D_ES_combined_on_ES")
results.append(res_d_es)


Evaluating Model A on English test
A_EN_on_EN -> evaluados 0/544
A_EN_on_EN -> evaluados 100/544
A_EN_on_EN -> evaluados 200/544
A_EN_on_EN -> evaluados 300/544
A_EN_on_EN -> evaluados 400/544
A_EN_on_EN -> evaluados 500/544

Evaluating Model A on Spanish test (cross-lingual)
A_EN_on_ES -> evaluados 0/544
A_EN_on_ES -> evaluados 100/544
A_EN_on_ES -> evaluados 200/544
A_EN_on_ES -> evaluados 300/544
A_EN_on_ES -> evaluados 400/544
A_EN_on_ES -> evaluados 500/544

Evaluating Model B on Spanish test
B_ES_translated_on_ES -> evaluados 0/544
B_ES_translated_on_ES -> evaluados 100/544
B_ES_translated_on_ES -> evaluados 200/544
B_ES_translated_on_ES -> evaluados 300/544
B_ES_translated_on_ES -> evaluados 400/544
B_ES_translated_on_ES -> evaluados 500/544

Evaluating Model C on Spanish test
C_ES_synthetic_on_ES -> evaluados 0/544
C_ES_synthetic_on_ES -> evaluados 100/544
C_ES_synthetic_on_ES -> evaluados 200/544
C_ES_synthetic_on_ES -> evaluados 300/544
C_ES_synthetic_on_ES -> evaluados 400/

In [ ]:
results_df = pd.DataFrame([
    {
        "model": r["label"],
        "precision": round(r["precision"], 4),
        "recall": round(r["recall"], 4),
        "f1": round(r["f1"], 4),
        "tp": r["tp"],
        "fp": r["fp"],
        "fn": r["fn"]
    }
    for r in results
]).sort_values("f1", ascending=False)

results_df

,model,precision,recall,f1,tp,fp,fn
0,A_EN_on_EN,0.5517,0.4369,0.4877,336,273,433
4,D_ES_combined_on_ES,0.4942,0.3958,0.4396,300,307,458
2,B_ES_translated_on_ES,0.4889,0.3760,0.4251,285,298,473
1,A_EN_on_ES,0.3286,0.2467,0.2818,187,382,571
3,C_ES_synthetic_on_ES,0.0842,0.0660,0.0740,50,544,708


In [ ]:
print("=== RESULTS TABLE ===")
for _, row in results_df.iterrows():
    print(
        f"{row['model']:28s} | "
        f"P={row['precision']:.4f} | "
        f"R={row['recall']:.4f} | "
        f"F1={row['f1']:.4f} | "
        f"TP={int(row['tp'])} | FP={int(row['fp'])} | FN={int(row['fn'])}"
    )

b_f1 = float(results_df.loc[results_df["model"] == "B_ES_translated_on_ES", "f1"].iloc[0])
c_f1 = float(results_df.loc[results_df["model"] == "C_ES_synthetic_on_ES", "f1"].iloc[0])
d_f1 = float(results_df.loc[results_df["model"] == "D_ES_combined_on_ES", "f1"].iloc[0])

print("\nKey comparisons:")
if c_f1 > b_f1:
    print(f"Synthetic beats translated by {c_f1 - b_f1:.4f} F1")
else:
    print(f"Translated beats synthetic by {b_f1 - c_f1:.4f} F1")

if d_f1 > max(b_f1, c_f1):
    print(f"Combined is best Spanish setup by {d_f1 - max(b_f1, c_f1):.4f} F1")
else:
    print("Combined does not beat the best single-source Spanish setup")

=== RESULTS TABLE ===
A_EN_on_EN                   | P=0.5517 | R=0.4369 | F1=0.4877 | TP=336 | FP=273 | FN=433
D_ES_combined_on_ES          | P=0.4942 | R=0.3958 | F1=0.4396 | TP=300 | FP=307 | FN=458
B_ES_translated_on_ES        | P=0.4889 | R=0.3760 | F1=0.4251 | TP=285 | FP=298 | FN=473
A_EN_on_ES                   | P=0.3286 | R=0.2467 | F1=0.2818 | TP=187 | FP=382 | FN=571
C_ES_synthetic_on_ES         | P=0.0842 | R=0.0660 | F1=0.0740 | TP=50 | FP=544 | FN=708

Key comparisons:
Translated beats synthetic by 0.3511 F1
Combined is best Spanish setup by 0.0145 F1


In [ ]:
best_spanish = max(
    [res_b_es, res_c_es, res_d_es],
    key=lambda x: x["f1"]
)

print("Best Spanish model:", best_spanish["label"])
for i, ex in enumerate(best_spanish["debug_examples"], 1):
    print("=" * 100)
    print("Ejemplo", i)
    print("INPUT:", ex["input"])
    print("GOLD:", ex["gold"])
    print("PRED:", ex["pred"])
    print("RAW :", ex["raw_pred"])

Best Spanish model: D_ES_combined_on_ES
Ejemplo 1
INPUT: extract triplets: Esperé entre 10 y 15 minutos para servicio, pedí una cerveza y nunca más me atendieron.
GOLD: [('servicio', 'servicegeneral', 'negative')]
PRED: [('cerveza', 'drinksquality', 'positive')]
RAW : [TERM] cerveza [CAT] drinksquality [POL] positive
Ejemplo 2
INPUT: extract triplets: El ambiente era un descanso tranquilo y relajante entre todos los niños corriendo por Downtown Disney.
GOLD: [('ambiente', 'ambiencegeneral', 'positive')]
PRED: [('ambiente', 'ambiencegeneral', 'positive')]
RAW : [TERM] ambiente [CAT] ambiencegeneral [POL] positive [SEP] [TERM] ambiente [CAT] ambiencegeneral [POL] positive
Ejemplo 3
INPUT: extract triplets: pequeña joya escondida
GOLD: [('NULL', 'restaurantgeneral', 'positive')]
PRED: [('NULL', 'restaurantgeneral', 'positive')]
RAW : [TERM] NULL [CAT] restaurantgeneral [POL] positive
Ejemplo 4
INPUT: extract triplets: Fui allí con un amigo de fuera de la ciudad... ¡y quedamos muy impresio

In [ ]:
def predict_triplets(model, review_text):
    model.eval()

    inputs = tokenizer(
        f"extract triplets: {review_text}",
        return_tensors="pt",
        max_length=cfg.max_source_length,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=cfg.max_target_length,
            num_beams=4
        )

    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    pred_triplets = parse_generated_triplets(pred_text)
    return pred_text, pred_triplets

sample_text = "La habitación estaba sucia pero el personal fue muy amable y la ubicación era excelente."
raw_pred, triplets = predict_triplets(model_d, sample_text)

print("Texto:", sample_text)
print("Raw:", raw_pred)
print("Triplets:", triplets)

Texto: La habitación estaba sucia pero el personal fue muy amable y la ubicación era excelente.
Raw: [TERM] habitación [CAT] roomsdesign_features [POL] positive [SEP] [TERM] personal [CAT] servicegeneral [POL] positive
Triplets: [('habitación', 'roomsdesign_features', 'positive'), ('personal', 'servicegeneral', 'positive')]


In [ ]:
results_csv_path = os.path.join(cfg.output_root, "results_comparison.csv")
results_df.to_csv(results_csv_path, index=False)
print("Saved:", results_csv_path)

Saved: /kaggle/working/absa_models/results_comparison.csv


In [ ]:
import shutil

shutil.make_archive("/kaggle/working/output_all", "zip", "/kaggle/working")
print("Creado /kaggle/working/output_all.zip")

Creado /kaggle/working/output_all.zip
